In [ ]:
!pip install torchinfo

In [ ]:
import random, numpy as np, torch
import torch
import torch.nn as nn
import torch.nn.functional as F
import nltk

seed = 42

# Python / NumPy
random.seed(seed)
np.random.seed(seed)

# Torch CPU/GPU
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# CuDNN determinism
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

[nltk_data] Downloading package punkt to C:\Users\Dinh
[nltk_data]     Vinh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
nltk.download('punkt')

# 1. Data preparation

In [10]:
from datasets import load_dataset

imdb = load_dataset("imdb")
train_data, test_data = imdb['train'], imdb['test']

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [11]:
# test
print(train_data.shape)
print(test_data.shape)

(25000, 2)
(25000, 2)


In [24]:
print(train_data[0]['text'])

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

In [25]:
print(train_data[0]['label'])

0


In [ ]:
# !pip -q install tokenizers

from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase

hf_tokenizer = Tokenizer(WordLevel(unk_token="<unk>"))
hf_tokenizer.normalizer = Lowercase()
hf_tokenizer.pre_tokenizer = Whitespace()

vocab_limit = 20000
min_freq = 3

trainer = WordLevelTrainer(
    vocab_size=vocab_limit,
    min_frequency=min_freq,
    special_tokens=["<pad>", "</s>", "<unk>", "<s>"]
)

hf_tokenizer.train_from_iterator((x["text"] for x in train_data), trainer)

vocab = hf_tokenizer.get_vocab()
vocab_ordered = dict(sorted(vocab.items(), key=lambda x: x[1]))

PAD_ID = hf_tokenizer.token_to_id("<pad>")
EOS_ID = hf_tokenizer.token_to_id("</s>")
UNK_ID = hf_tokenizer.token_to_id("<unk>")
SOS_ID = hf_tokenizer.token_to_id("<s>")
assert SOS_ID is not None

vocab_size = len(vocab_ordered)

print("HF vocab_size =", vocab_size)
print("SPECIAL:", {"<pad>": PAD_ID, "</s>": EOS_ID, "<unk>": UNK_ID, "<s>": SOS_ID})


In [ ]:
seq_length = 500

def text_pipeline(text: str):
    return hf_tokenizer.encode(text).ids

def collate_batch_pad_before(batch, seq_length=seq_length):
    if seq_length < 1:
        raise ValueError("seq_length must be positive")
    text_list, label_list = [], []
    for sample in batch:
        label_list.append(sample["label"])

        # Reserve one position for the start token, including long reviews.
        ids = text_pipeline(sample["text"])[:seq_length - 1]
        pad_size = seq_length - len(ids) - 1
        ids = [PAD_ID] * pad_size + [SOS_ID] + ids

        text_list.append(ids)

    return (torch.tensor(text_list, dtype=torch.int64),
            torch.tensor(label_list, dtype=torch.int64))


In [ ]:
from torch.utils.data import DataLoader
batch_size = 64

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_batch_pad_before, drop_last=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, collate_fn=collate_batch_pad_before, drop_last=True)

In [ ]:
for inputs, labels in train_loader:
    sample = inputs[0]
    print(sample)
    break

# 2. Model

In [ ]:
class TextClsModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.rnn = nn.RNN(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x)
        _, hidden = self.rnn(x)
        last_hidden = hidden[-1,:,:] 
        x = self.fc(last_hidden)
        return x


In [ ]:
model = TextClsModel(vocab_size=vocab_size, 
                     emb_dim=128, 
                     hidden_dim=64,
                     num_layers=2
                     )

In [ ]:
from torchsummary import summary

random_tensor = torch.randint(low=0, high=vocab_size, size=(64, seq_length), dtype=torch.long)
print('random_tensor:', random_tensor.shape)
summary(model, random_tensor)

# 3. Evaluate

In [ ]:
def evaluate(model, test_loader, criterion, device):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad(): 
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            test_loss += loss.item()
            correct +=(predicted == labels).sum().item()

    test_loss = test_loss / len(test_loader)
    accuracy = 100* correct / total

    return test_loss, accuracy

# 4. Train

In [ ]:
device = 'cuda'
max_epoch = 20
LR = 0.01
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [ ]:
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

model.to(device)
for epoch in range(max_epoch):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        running_correct += (predicted == labels).sum().item()

        loss.backward()
        optimizer.step()

    epoch_loss =  running_loss / len(train_loader)
    epoch_accuracy = 100* running_correct / total
    test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)

    print(f"Epoch [{epoch + 1}/{max_epoch}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%, Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)
    test_losses.append(test_loss)
    test_accuracies.append(test_accuracy)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(train_accuracies, label='Training Accuracy')
plt.plot(test_accuracies, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([min(plt.ylim()),100])
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(train_losses, label='Training Loss')
plt.plot(test_losses, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Binary Cross Entropy')
plt.title('Training and Validation Loss')
plt.xlabel('epoch')
plt.show()